In [5]:
!pip install chromadb sentence-transformers pandas -q --use-deprecated=legacy-resolver

In [6]:
%%writefile data_loader.py
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def load_and_clean_data(filepath: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(filepath)
        logging.info(f"Successfully loaded {len(df)} rows from {filepath}")
        df = df.dropna(subset=["question", "answer"])
        df["question"] = df["question"].str.strip()
        df["answer"] = df["answer"].str.strip()
        df["search_chunk"] = "Category: " + df["category"] + " | Q: " + df["question"] + " | A: " + df["answer"]
        return df
    except Exception as e:
        logging.error(f"Error: {e}")
        raise

Writing data_loader.py


In [7]:
%%writefile search_engine.py
import chromadb
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class FAQSearchEngine:
    def __init__(self, collection_name="aastu_faqs"):
        self.client = chromadb.Client()
        self.collection = self.client.create_collection(name=collection_name)
        logging.info("Search Engine initialized.")

    def index_documents(self, documents: list, ids: list):
        self.collection.add(documents=documents, ids=ids)
        logging.info(f"Indexed {len(documents)} documents.")

    def search(self, query: str, top_k: int = 3):
        results = self.collection.query(query_texts=[query], n_results=top_k)
        return results["documents"][0], results["distances"][0]

Writing search_engine.py


In [8]:
%%writefile requirements.txt
pandas
chromadb
sentence-transformers

Writing requirements.txt


In [9]:
from data_loader import load_and_clean_data
from search_engine import FAQSearchEngine

# 1. Load and clean the data
print("Loading data...")
df = load_and_clean_data("data/aastu_faqs.csv")

# 2. Prepare the data for the database
documents = df["search_chunk"].tolist()
ids = [f"doc_{i}" for i in range(len(documents))]

# 3. Initialize and index in ChromaDB
print("Initializing search engine and generating embeddings...")
engine = FAQSearchEngine()
engine.index_documents(documents, ids)

# 4. Query the database
query = "I can't log into the student portal to pick my courses."
matches, scores = engine.search(query, top_k=3)

print(f"\n🔍 Query: {query}")
print("="*60)
for i, (match, score) in enumerate(zip(matches, scores)):
    print(f"Match {i+1} (Distance: {score:.4f}):\n{match}\n" + "-"*60)

Loading data...
Initializing search engine and generating embeddings...


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 42.2MiB/s]



🔍 Query: I can't log into the student portal to pick my courses.
Match 1 (Distance: 0.8372):
Category: Registration | Q: How do I register for my classes? | A: Log into portal.aastu.edu.et using your student ID. Navigate to the 'Registration' tab. Registration opens two weeks before the semester begins.
------------------------------------------------------------
Match 2 (Distance: 1.0642):
Category: IT Support | Q: How do I connect to the campus Wi-Fi? | A: The student Wi-Fi network is 'AASTU-Student'. Use your student ID number as the default password.
------------------------------------------------------------
Match 3 (Distance: 1.4232):
Category: Finance | Q: How do I pay my tuition fees? | A: Tuition fees can be paid at the finance office or via the CBE mobile app using your assigned student account number.
------------------------------------------------------------
